# Trove newspaper counts refresh (DV2)

Gets **counts only** (metadata, allowed by the Trove API Terms of Use): total digitised newspaper articles,
articles by state, articles by year of publication, and articles with text corrections.
About 26 API calls, one per second (the quota is 200 per minute).

**Your API key is typed in when asked. It is never saved in this notebook, the output files or the repository.**
Same settings as Tim Sherratt's harvester (`category=newspaper`, `l-artType=newspaper`), so the numbers compare with the February 2025 data.
Years with no articles are left out, never set to zero.

In [ ]:
from getpass import getpass
import time, datetime
import requests
import pandas as pd

trove_api_key = getpass("Paste your Trove API key, then press Enter: ")
session = requests.Session()
session.headers["X-API-KEY"] = trove_api_key          # sent in a header, so it never appears in a printed URL
api_calls_made = 0

def trove_search(extra_parameters):
    global api_calls_made
    parameters = {"category": "newspaper", "l-artType": "newspaper", "encoding": "json", "n": 0}
    parameters.update(extra_parameters)
    response = session.get("https://api.trove.nla.gov.au/v3/result", params=parameters, timeout=60)
    api_calls_made += 1
    response.raise_for_status()
    time.sleep(1)
    return response.json()

def facet_terms(response_data):
    terms = []
    for facet in response_data["category"][0].get("facets", {}).get("facet", []):
        for term in facet["term"]:
            terms.append({"term": term["search"], "display": term.get("display", term["search"]), "total": int(term["count"])})
    return terms

harvest_time = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
print("Harvest started", harvest_time)

In [ ]:
# 1) Overall totals
all_articles = trove_search({})["category"][0]["records"]["total"]
corrected_articles = trove_search({"q": "has:corrections"})["category"][0]["records"]["total"]
totals = pd.DataFrame([{"harvest_type": "all", "total": all_articles},
                       {"harvest_type": "has:corrections", "total": corrected_articles}])
print(totals)

In [ ]:
# 2) Articles by state or territory (one call, state facet)
states = pd.DataFrame(facet_terms(trove_search({"facet": "state"})))
states = states.rename(columns={"display": "state"})[["state", "total"]].sort_values("total", ascending=False)
print(states.to_string(index=False))
print("Sum of states:", f"{states.total.sum():,}", "| all articles:", f"{all_articles:,}")

In [ ]:
# 3) Articles by year of publication (year facet needs a decade limit: decades 180 to 202)
year_rows = []
for decade in range(180, 203):
    for term in facet_terms(trove_search({"facet": "year", "l-decade": decade})):
        year_rows.append({"year": int(term["term"]), "total": term["total"]})
years = pd.DataFrame(year_rows).drop_duplicates("year").sort_values("year")
print(len(years), "years with articles, from", years.year.min(), "to", years.year.max())
print("Sum of years:", f"{years.total.sum():,}", "| all articles:", f"{all_articles:,}")
print(years[years.year.between(1950, 1960)].to_string(index=False))

In [ ]:
# 4) Save the counts and a log, then download them
totals.to_csv("trove_totals_refresh.csv", index=False)
states.to_csv("trove_states_refresh.csv", index=False)
years.to_csv("trove_by_year_refresh.csv", index=False)
with open("trove_refresh_log.txt", "w") as log_file:
    log_file.write(f"Harvest time: {harvest_time}\n")
    log_file.write("Parameters: category=newspaper, l-artType=newspaper (same as Tim Sherratt's harvester)\n")
    log_file.write(f"API calls made: {api_calls_made}\n")
    log_file.write(f"All articles: {all_articles}\nArticles with corrections: {corrected_articles}\n")
    log_file.write(f"Sum of states: {states.total.sum()}\nSum of years: {years.total.sum()}\n")
del trove_api_key, session                             # forget the key
from google.colab import files
for output_name in ["trove_totals_refresh.csv", "trove_states_refresh.csv", "trove_by_year_refresh.csv", "trove_refresh_log.txt"]:
    files.download(output_name)